In [5]:
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.optimizers import Adam
import pandas as pd 
import tensorflow as tf
from tensorflow.keras.regularizers import l2
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,accuracy_score
from keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.utils import to_categorical


In [6]:
df = pd.read_csv('D:\Let Me Cook\DPL302m\Building_your_Deep_Neural_Network_Step_by_Step\AKH_WQI.csv')
df.head()

,PH,Temp,Turbidity,TSS,BOD5,COD,DO,Amoni,Phosphat,Coliforms,WQI
0,7.2,27.8,60.0,210.0,3.20,6.6,7.2,0.35,0.84,2600,49.68
1,7.1,27.6,65.0,180.0,3.15,6.8,7.7,0.88,0.95,4700,65.34
2,7.3,27.7,65.0,195.0,3.68,7.8,7.0,2.25,1.36,8500,51.53
3,7.1,27.5,70.0,180.0,3.50,6.2,6.7,2.33,1.41,7500,55.26
4,6.9,27.5,80.0,100.0,3.84,6.8,6.4,1.87,2.74,9500,49.51


In [7]:
scaler = StandardScaler()
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

X = scaler.fit_transform(X)

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [9]:
model = Sequential()
model.add(Dense(units=128, activation="relu", input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=64, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=32, activation="relu", kernel_regularizer=l2(0.01)))
model.add(Dropout(0.2))
model.add(Dense(units=1, activation="relu", kernel_regularizer=l2(0.01)))
          
model.compile(optimizer=Adam(learning_rate=0.01), loss='mean_squared_error', metrics=['mae'])
          
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               1408      
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dropout_2 (Dropout)         (None, 32)                0         
                                                                 
 dense_3 (Dense)             (None, 1)                 3

In [10]:
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

model.fit(X_train, y_train, epochs=100, batch_size=20, validation_data=(X_val, y_val))

Epoch 1/100
23/23 [==============================] - 3s 16ms/step - loss: 1069.7782 - mae: 25.6391 - val_loss: 662.0402 - val_mae: 21.8804
Epoch 2/100
23/23 [==============================] - 0s 6ms/step - loss: 435.9542 - mae: 16.4318 - val_loss: 376.7629 - val_mae: 16.3448
Epoch 3/100
23/23 [==============================] - 0s 6ms/step - loss: 369.8608 - mae: 15.0947 - val_loss: 315.1072 - val_mae: 14.5189
Epoch 4/100
23/23 [==============================] - 0s 6ms/step - loss: 347.1712 - mae: 14.5828 - val_loss: 291.7975 - val_mae: 13.6943
Epoch 5/100
23/23 [==============================] - 0s 6ms/step - loss: 414.6599 - mae: 15.7032 - val_loss: 292.2662 - val_mae: 13.2504
Epoch 6/100
23/23 [==============================] - 0s 6ms/step - loss: 352.4441 - mae: 14.6737 - val_loss: 278.1344 - val_mae: 13.3051
Epoch 7/100
23/23 [==============================] - 0s 6ms/step - loss: 341.3475 - mae: 14.7048 - val_loss: 343.7079 - val_mae: 15.4583
Epoch 8/100
23/23 [====================

In [11]:
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {train_mse}")
print(f"Validation MSE: {val_mse}")
print(f"testing MSE: {test_mse}")

4/4 [==============================] - 0s 3ms/step
Train MSE: 107.43273483445398
Validation MSE: 264.8308695291809
testing MSE: 277.71628871173937


Với câu trúc tự define hôm trước MSE(trainig) < MSE (testing) ==> Hight Variance(overfitting)

thay đổi cấu trúc ANN bằng cách giảm sô nơ ron trong một layer bằng cách dropout và sử dụng L2(lasso) kèm với việc sử dụng early stoping thay đổi cấu trúc của mạng ANN liên tục thì MSE trên cả 3 tập có sự thay đôi tố hơn nhưng Train MSE(119.106) < testing MSE(278.3795). Tuy kết quả có tốt hơn nhiều bằng cách sử dụng các phương pháp nhằm cải thiện variance nhưng model trên vẫn bị overfitting.